# HAGI - GDR Geo-Ablation Diagnostic (Kaggle T4, one shot)

GDR (model D) lost to recurrence-only (model B) on held-out loss, 0/5 seeds. This
isolates **whether the geometric-product term is the culprit**, by running three
matched variants on the SAME held-out shard:

- **B** - recurrence only (no GDR)
- **D** - full GDR (reproduces the negative on this platform = control)
- **D_nogeo** - GDR with the geometric product OFF (same 114.63M params as D)

**Read:** if `D_nogeo` recovers toward `B`, the geometric product was the noise;
if `D_nogeo` is still worse than `B`, the grade machinery itself hurts.

## Kaggle settings (do this first)
1. **Settings -> Accelerator -> GPU T4 x2** (we use one T4).
2. **Settings -> Internet -> On** (needed for git clone, pip, tokenizing FineWeb-Edu).
3. Run top to bottom. The **smoke** cell (5) catches OOM/NaN cheaply before the long run.

T4 has **no real bf16** - everything here runs **fp16** (the loop uses a GradScaler).
Designed to fit one ~10h session (default 2 seeds). Resumable within the session
(re-run cell 6; finished runs skip). To pause across sessions, use **Save Version**.

## 1. Clone + update the repo

In [1]:
import os
%cd /kaggle/working
if not os.path.isdir('HAGI'):
    !git clone -b experimental https://github.com/ShmidtS/HAGI.git
%cd /kaggle/working/HAGI
!git pull --ff-only origin experimental   # needs --precision/--seed/--no-compile + geo flag
print('cwd:', os.getcwd())

/kaggle/working
Cloning into 'HAGI'...
remote: Enumerating objects: 1988, done.
remote: Counting objects: 100% (808/808), done.
remote: Compressing objects: 100% (356/356), done.
remote: Total 1988 (delta 546), reused 631 (delta 399), pack-reused 1180 (from 1)
Receiving objects: 100% (1988/1988), 50.28 MiB | 26.63 MiB/s, done.
Resolving deltas: 100% (1058/1058), done.
/kaggle/working/HAGI
From https://github.com/ShmidtS/HAGI
 * branch            experimental -> FETCH_HEAD
Already up to date.
cwd: /kaggle/working/HAGI


## 2. Install extras (Kaggle already has torch/numpy/pyyaml - don't touch torch)

In [2]:
!pip install -q -U datasets transformers huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 2.7 MB/s eta 0:00:00a 0:00:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 7.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 684.4/684.4 kB 13.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.6/116.6 kB 5.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.


## 3. Data + held-out split
Pulls tokenized shards from an **HF dataset repo** if you set `DATA_REPO` below
(recommended - same corpus everywhere, no re-tokenizing); else uses uploaded `.bin`
shards under `/kaggle/input`; else tokenizes FineWeb-Edu fresh (~20 min). Then holds
out the **last shard** as validation. Every path requires the **SmolLM2** tokenizer
(vocab 49,152) - other tokenizers are incompatible.

In [3]:
import glob, os
# Data source priority: HF dataset repo (DATA_REPO) > uploaded Kaggle Dataset > tokenize fresh.
DATA_REPO = 'NAME0x0/hagi-fineweb-edu-smollm2'   # e.g. 'NAME0x0/hagi-fineweb-edu-smollm2' to pull tokenized shards from HF (recommended)
if DATA_REPO:
    from huggingface_hub import snapshot_download
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ.setdefault('HF_TOKEN', UserSecretsClient().get_secret('HF_TOKEN'))
    except Exception:
        pass  # a public dataset needs no token
    SRC = snapshot_download(repo_id=DATA_REPO, repo_type='dataset', allow_patterns='*.bin',
                            token=os.environ.get('HF_TOKEN'))
    print('pulled shards from HF dataset', DATA_REPO, '->', SRC)
else:
    up = glob.glob('/kaggle/input/*/*.bin')
    if up:
        SRC = os.path.dirname(up[0]); print('using uploaded shards:', SRC)
    else:
        SRC = '/kaggle/working/data'
        if not glob.glob(f'{SRC}/*.bin'):
            print('no uploaded shards -> tokenizing FineWeb-Edu (~20 min, one time)...')
            !python -m prototype.data.tokenize --dataset HuggingFaceFW/fineweb-edu --subset sample-10BT \
                --output {SRC} --tokenizer HuggingFaceTB/SmolLM2-135M --limit 600000
        print('shards dir:', SRC)

shards = sorted(glob.glob(f'{SRC}/*.bin'))
assert len(shards) >= 2, f'need >=2 shards, found {len(shards)} in {SRC}'
TRAIN_DIR, VAL_DIR = '/kaggle/working/train', '/kaggle/working/val'
for d in (TRAIN_DIR, VAL_DIR):
    os.makedirs(d, exist_ok=True)
    for f in glob.glob(f'{d}/*.bin'): os.remove(f)
for s in shards[:-1]: os.symlink(s, f'{TRAIN_DIR}/{os.path.basename(s)}')
os.symlink(shards[-1], f'{VAL_DIR}/{os.path.basename(shards[-1])}')
VAL_NAME = os.path.basename(shards[-1])
print(f'train: {len(shards)-1} shards | held-out val: {VAL_NAME}')

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

pulled shards from HF dataset NAME0x0/hagi-fineweb-edu-smollm2 -> /root/.cache/huggingface/hub/datasets--NAME0x0--hagi-fineweb-edu-smollm2/snapshots/df215c2f29c63a1dbacea0016100bd2bddc7bf19
train: 6 shards | held-out val: shard_00006.bin


## 4. Knobs + preflight
Raise `SEEDS` to `[1,2,3]` only if you'll have a second session. Read the time
estimate vs the 12h Kaggle limit before running.

In [4]:
import torch
# ---- knobs ----
SEEDS  = [1]               # 2 seeds fits one ~10h session; 3 needs a second
TOKENS = 120_000_000          # per run (matches the prior held-out experiment)
WORK   = '/kaggle/working/ckpt'
# ---------------
VARIANTS = ['b', 'd', 'd_nogeo']
TPS    = 4 * 16 * 1024        # batch 4 * accum 16 * seq 1024 = 65,536 (matched eff batch)
STEPS  = TOKENS // TPS + 50
MAXSTEP = -(-TOKENS // TPS)
os.makedirs(WORK, exist_ok=True)

assert torch.cuda.is_available(), 'No GPU. Settings -> Accelerator -> GPU T4 x2.'
p = torch.cuda.get_device_properties(0)
print(f'GPU: {p.name} | {p.total_memory/1e9:.0f} GB | running fp16 (T4 has no real bf16)')
assert glob.glob(f'{TRAIN_DIR}/*.bin') and glob.glob(f'{VAL_DIR}/*.bin'), 'split missing - rerun cell 3'

runs = len(SEEDS) * len(VARIANTS)
est_h = runs * (TOKENS / 15000 / 60) / 60   # ~15k tok/s, conservative
print(f'plan: {runs} runs ({len(VARIANTS)} variants x {len(SEEDS)} seeds) @ {TOKENS:,} tok/run')
print(f'EST: ~{est_h:.1f} h (rough; smoke prints REAL tok/s next). Kaggle session cap = 12h.')

GPU: Tesla T4 | 16 GB | running fp16 (T4 has no real bf16)
plan: 3 runs (3 variants x 1 seeds) @ 120,000,000 tok/run
EST: ~6.7 h (rough; smoke prints REAL tok/s next). Kaggle session cap = 12h.


## 5. Smoke test (8 steps per variant, ~5-10 min)
Catches OOM / NaN / compile errors **before** the long run. If a variant OOMs,
lower `--batch-size` to 2 in cells 5-6; if you see a compile/inductor error, add
`--no-compile`; if loss is `nan`, switch `--precision fp16` -> `bf16` (slower).

In [5]:
import subprocess
print('SMOKE: 8 steps each variant (fp16, batch 4).\n')
for v in VARIANTS:
    print(f'--- smoke {v} ---', flush=True)
    r = subprocess.run(['python','-u','-m','prototype.training.train',
        '--config', f'configs/ablation_{v}.yaml', '--data', TRAIN_DIR, '--device','cuda',
        '--ckpt-dir','/kaggle/working/smoke', '--precision','fp16','--batch-size','4',
        '--grad-accum-steps','16','--train-tokens','1000000','--steps','8'], cwd='.')
    assert r.returncode == 0, f'smoke {v} FAILED - read the traceback above'
    assert glob.glob(f'/kaggle/working/smoke/ablation_{v}/step-*.pt'), f'{v}: no checkpoint'
    print(f'  smoke {v} OK\n')
print('ALL SMOKE OK - safe to run cell 6.')

SMOKE: 8 steps each variant (fp16, batch 4).

--- smoke b ---
precision overridden -> fp16
batch_size overridden -> 4
grad_accum_steps overridden -> 16
train_tokens overridden -> 1,000,000
[ablation_b] parameters: 113.3M
torch.compile enabled
max_steps=16 derived from train_tokens=1,000,000 (65,536 tokens/step)
session: steps 0..8 (then checkpoint + exit)


W0609 20:07:43.664000 168 torch/_inductor/utils.py:1679] [0/0] Not enough SMs to use max_autotune_gemm mode


step      0 | loss 10.9578 | lr 7.50e-07 | 494 tok/s
checkpoint -> /kaggle/working/smoke/ablation_b/step-00000008.pt
  smoke b OK

--- smoke d ---
precision overridden -> fp16
batch_size overridden -> 4
grad_accum_steps overridden -> 16
train_tokens overridden -> 1,000,000
[ablation_d] parameters: 114.6M
torch.compile enabled
max_steps=16 derived from train_tokens=1,000,000 (65,536 tokens/step)
session: steps 0..8 (then checkpoint + exit)


W0609 20:10:48.060000 348 torch/_inductor/utils.py:1679] [0/0] Not enough SMs to use max_autotune_gemm mode
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/sympy/core/assumptions.py", line 499, in getit
    return self._assumptions[fact]
           ~~~~~~~~~~~~~~~~~^^^^^^
KeyError: 'extended_nonnegative'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/kaggle/working/HAGI/prototype/training/train.py", line 165, in <module>
    main()
  File "/kaggle/working/HAGI/prototype/training/train.py", line 159, in main
    train(model, optimizer, get_batch, loop_cfg, device=args.device,
  File "/kaggle/working/HAGI/prototype/training/loop.py", line 114, in train
    _, loss = model(x, targets=y)
              ^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval

KeyboardInterrupt: 

## 6. Run all variants x seeds, eval on held-out, record
Resumable: re-run this cell to continue; completed runs are skipped. Output streams
live (`-u`).

In [6]:
import subprocess, json, re

def done(d):
    cks = glob.glob(f'{d}/step-*.pt')
    return bool(cks) and max(int(re.search(r'step-(\d+)', c).group(1)) for c in cks) >= MAXSTEP

def train(v, seed):
    d = f'{WORK}/s{seed}/ablation_{v}'
    if done(d): print(f'  [{v} s{seed}] complete -> skip'); return
    subprocess.run(['python','-u','-m','prototype.training.train',
        '--config', f'configs/ablation_{v}.yaml', '--data', TRAIN_DIR, '--device','cuda',
        '--ckpt-dir', f'{WORK}/s{seed}', '--seed', str(seed), '--precision','fp16',
        '--batch-size','4','--grad-accum-steps','16','--train-tokens',str(TOKENS),
        '--steps',str(STEPS),'--resume','auto'], cwd='.', check=True)

def latest(v, seed):
    cks = sorted(glob.glob(f'{WORK}/s{seed}/ablation_{v}/step-*.pt'))
    return cks[-1] if cks else None

RES = '/kaggle/working/geo_results.json'
results = json.load(open(RES)) if os.path.exists(RES) else {}
for seed in SEEDS:
    print(f'\n========== SEED {seed} ==========', flush=True)
    for v in VARIANTS: train(v, seed)
    cks = [latest(v, seed) for v in VARIANTS]
    r = subprocess.run(['python','-u','scripts/eval_loss.py','--data',VAL_DIR,'--device','cuda',
        '--batches','50','--json','--ckpt', *cks], cwd='.', capture_output=True, text=True)
    print(r.stdout)
    line = next((l for l in r.stdout.splitlines() if l.startswith('JSON ')), None)
    if line:
        j = json.loads(line[5:])
        results[str(seed)] = {'b': j['ablation_b']['loss'], 'd': j['ablation_d']['loss'],
                              'dng': j['ablation_d_nogeo']['loss']}
        json.dump(results, open(RES, 'w'), indent=2)
    else:
        print('STDERR', r.stderr)


========== SEED 1 ==========
precision overridden -> fp16
batch_size overridden -> 4
grad_accum_steps overridden -> 16
train_tokens overridden -> 120,000,000
[ablation_b] parameters: 113.3M
--resume auto: no checkpoint found, starting fresh
torch.compile enabled
max_steps=1832 derived from train_tokens=120,000,000 (65,536 tokens/step)
session: steps 0..1881 (then checkpoint + exit)
step      0 | loss 10.9659 | lr 7.50e-07 | 4,470 tok/s
step     50 | loss 9.1296 | lr 3.83e-05 | 11,753 tok/s
step    100 | loss 7.5218 | lr 7.57e-05 | 11,905 tok/s
step    150 | loss 6.8748 | lr 1.13e-04 | 11,724 tok/s
step    200 | loss 6.4121 | lr 1.51e-04 | 11,685 tok/s
step    250 | loss 6.1634 | lr 1.88e-04 | 11,656 tok/s
step    300 | loss 5.8970 | lr 2.26e-04 | 11,623 tok/s
step    350 | loss 5.7002 | lr 2.63e-04 | 11,594 tok/s
step    400 | loss 5.6177 | lr 3.00e-04 | 11,606 tok/s
step    450 | loss 5.4273 | lr 2.99e-04 | 11,625 tok/s
step    500 | loss 5.2893 | lr 2.97e-04 | 11,640 tok/s
checkpoin

W0609 23:04:00.816000 473 torch/_inductor/utils.py:1679] [0/0] Not enough SMs to use max_autotune_gemm mode


step      0 | loss 10.9810 | lr 7.50e-07 | 448 tok/s
step     50 | loss 9.1390 | lr 3.83e-05 | 11,374 tok/s
step    100 | loss 7.5116 | lr 7.57e-05 | 11,577 tok/s
step    150 | loss 6.8757 | lr 1.13e-04 | 11,357 tok/s
step    200 | loss 6.4294 | lr 1.51e-04 | 11,280 tok/s
step    250 | loss 6.2059 | lr 1.88e-04 | 11,299 tok/s
step    300 | loss 5.9407 | lr 2.26e-04 | 11,233 tok/s
step    350 | loss 5.7437 | lr 2.63e-04 | 11,219 tok/s
step    400 | loss 5.6619 | lr 3.00e-04 | 11,222 tok/s
step    450 | loss 5.4734 | lr 2.99e-04 | 11,222 tok/s
step    500 | loss 5.3177 | lr 2.97e-04 | 11,225 tok/s
checkpoint -> /kaggle/working/ckpt/s1/ablation_d/step-00000500.pt
step    550 | loss 5.2256 | lr 2.93e-04 | 11,185 tok/s
step    600 | loss 5.1186 | lr 2.87e-04 | 11,271 tok/s
step    650 | loss 5.1056 | lr 2.80e-04 | 11,276 tok/s
step    700 | loss 5.0328 | lr 2.72e-04 | 11,287 tok/s
step    750 | loss 4.9160 | lr 2.62e-04 | 11,216 tok/s
step    800 | loss 4.8151 | lr 2.51e-04 | 11,238 tok/s
s

W0610 02:04:09.353000 658 torch/_inductor/utils.py:1679] [0/0] Not enough SMs to use max_autotune_gemm mode


step      0 | loss 10.9812 | lr 7.50e-07 | 453 tok/s
step     50 | loss 9.1385 | lr 3.83e-05 | 11,483 tok/s
step    100 | loss 7.5179 | lr 7.57e-05 | 11,648 tok/s
step    150 | loss 6.8746 | lr 1.13e-04 | 11,452 tok/s
step    200 | loss 6.4282 | lr 1.51e-04 | 11,382 tok/s
step    250 | loss 6.1964 | lr 1.88e-04 | 11,310 tok/s
step    300 | loss 5.9290 | lr 2.26e-04 | 11,354 tok/s
step    350 | loss 5.7427 | lr 2.63e-04 | 11,346 tok/s
step    400 | loss 5.6600 | lr 3.00e-04 | 11,341 tok/s
step    450 | loss 5.4681 | lr 2.99e-04 | 11,365 tok/s
step    500 | loss 5.3267 | lr 2.97e-04 | 11,326 tok/s
checkpoint -> /kaggle/working/ckpt/s1/ablation_d_nogeo/step-00000500.pt
step    550 | loss 5.2306 | lr 2.93e-04 | 11,266 tok/s
step    600 | loss 5.1230 | lr 2.87e-04 | 11,339 tok/s
step    650 | loss 5.1041 | lr 2.80e-04 | 11,341 tok/s
step    700 | loss 5.0308 | lr 2.72e-04 | 11,289 tok/s
step    750 | loss 4.9116 | lr 2.62e-04 | 11,300 tok/s
step    800 | loss 4.8090 | lr 2.51e-04 | 11,323 t

## 7. Verdict

In [1]:
import statistics as st
print('============ GEO-DIAGNOSTIC SUMMARY ============')
print(f'held-out val: {VAL_NAME} | {TOKENS:,} tok/run | fp16 T4')
print(f"{'seed':>4} {'B':>8} {'D':>8} {'D_nogeo':>9} {'D-B':>8} {'Dng-B':>8}")
dnb, dngb = [], []
for s in SEEDS:
    r = results.get(str(s))
    if not r: print(f'{s:>4}  (missing)'); continue
    a, b, c = r['b'], r['d'], r['dng']
    dnb.append(b - a); dngb.append(c - a)
    print(f"{s:>4} {a:>8.4f} {b:>8.4f} {c:>9.4f} {b-a:>+8.4f} {c-a:>+8.4f}")
if dnb:
    mdnb, mdngb = st.mean(dnb), st.mean(dngb)
    print(f"\nmean D-B    = {mdnb:+.4f}  ({'D worse (reproduces negative)' if mdnb>0 else 'D better'})")
    print(f"mean Dng-B  = {mdngb:+.4f}  ({'D_nogeo worse than B' if mdngb>0 else 'D_nogeo >= B'})")
    print('\nREAD:')
    if mdnb > 0 and mdngb <= 0.25 * mdnb:
        print(' Geometric product is the main culprit - removing it recovers most of the gap.')
        print(' Next bet: a better (paper-faithful) geometric design, not grades-as-aux.')
    elif mdngb > 0:
        print(' Removing geo does NOT recover - the grade machinery itself hurts at this scale.')
        print(' Strong signal: GDR-as-built is the problem. Pivot (open models) or paper-faithful rebuild.')
    else:
        print(' D_nogeo beats B - grades without geo HELP. Unexpected; worth a focused follow-up.')
    print(f"\nPaste to Claude: GEO_DIAG seeds={SEEDS} mean_D-B={mdnb:+.4f} mean_Dnogeo-B={mdngb:+.4f}")

============ GEO-DIAGNOSTIC SUMMARY ============


NameError: name 'VAL_NAME' is not defined